In [4]:
import tushare as ts
import pandas as pd
import time

ts.set_token('13bb0841c8a377221b39d9142f42bae2e2e9a897b9f692c75dd90d65')
pro = ts.pro_api()

start_time = '2023-08-01 09:00:00'
end_time = '2023-08-31 15:00:00'
trade_date_for_list = '20230801'

print(f"正在获取 {trade_date_for_list} 的正常上市股票列表...")
df_stocks = pro.bak_basic(trade_date=trade_date_for_list, fields='ts_code,name,industry')
stock_list = df_stocks['ts_code'].tolist()


daily_data_dict = {}

for i, ts_code in enumerate(stock_list):
    try:
        df_min = pro.stk_mins(ts_code=ts_code, freq='1min', start_date=start_time, end_date=end_time)
        
        if df_min is not None and not df_min.empty:
            # 核心修改1：按 trade_time 升序排列并重置索引
            df_min = df_min.sort_values(by='trade_time', ascending=True).reset_index(drop=True)
            
            df_min['date_key'] = df_min['trade_time'].str[:10]
            
            for date, group in df_min.groupby('date_key'):
                if date not in daily_data_dict:
                    daily_data_dict[date] = []
                
                group = group.drop(columns=['date_key'])
                daily_data_dict[date].append(group)
                
    except Exception as e:
        print(f"请求 {ts_code} 时发生错误: {e}")
        
    if (i + 1) % 10 == 0:
        print(f"已处理 {i + 1} / {len(stock_list)} 只股票...")

    # 遵守 2次/分钟 的限制
    time.sleep(31)

print("\n开始按日期保存为 Parquet 文件...")
for date_key, dfs in daily_data_dict.items():
    if not dfs:
        continue
        
    df_daily = pd.concat(dfs, ignore_index=True)
    
    file_date = date_key.replace('-', '')
    file_name = f"/root/autodl-tmp/min_data/{file_date}.parquet"
    
    # 核心修改2：保存为 parquet 格式
    df_daily.to_parquet(file_name, engine='pyarrow')
    print(f"✅ 成功保存 {file_name}，包含 {len(df_daily)} 行数据。")

正在获取 20230801 的正常上市股票列表...
请求 000001.SZ 时发生错误: 抱歉，您每天最多访问该接口2次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。
请求 000002.SZ 时发生错误: 抱歉，您每天最多访问该接口2次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。


KeyboardInterrupt: 